# Rain or Pumps? Attributing India's Groundwater Decline District by District

Data exploration, preparation and feature engineering

Sivakumar Kakara, BITS Pilani PGCP AI/ML Capstone

**Objective:** build a model of the expected groundwater level per district given rainfall history alone, and treat the gap between expected and observed levels as a proxy for extraction pressure.

**Data:**
1. CGWB quality-controlled groundwater levels 2000-2022, 2759 wells with specific yield (figshare, doi:10.6084/m9.figshare.29293877.v3, CC BY 4.0)
2. IMD 0.25 degree gridded daily rainfall, 1998-2022 (imdpune.gov.in). 1998-99 are only used to compute lag features for the year 2000.

Background notes before touching the data. Recharge is dominated by the June-September monsoon; secondary sources such as canal seepage and irrigation return flow are not observed in our data. Consumption is dominated by irrigation and is also not observed, which is why the project uses a residual based design. Rainfall varies a lot from year to year and from place to place, and pumping tends to increase in exactly the years when recharge falls. So the plan is: put everything rainfall driven into the baseline model, and read the remaining systematic gap as extraction pressure.

## 1. Setup

In [1]:
import os, glob, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 110
RNG = 42

DATA   = Path('../data')
CGWB   = DATA / 'cgwb' / 'CGWB_India_filtered_GWLs_ref_sy_2000_2022.csv'
IMDDIR = DATA / 'imd_rainfall'
PROC   = DATA / 'processed'
PROC.mkdir(parents=True, exist_ok=True)
print('CGWB exists:', CGWB.exists(), '| IMD files:', len(list(IMDDIR.glob('imd_rf25_*.nc'))))

ModuleNotFoundError: No module named 'xarray'

## 2. Rainfall extraction and the spatial merge

The CGWB file has one row per well with seasonal reading columns (Jan/May/Aug/Nov campaigns, 2000-2022). The IMD files are yearly NetCDF grids of daily rainfall on a fixed 0.25 degree lattice. The two files share no key: IMD coordinates are lattice nodes, wells sit at arbitrary points, and only 3 of 2759 wells fall exactly on a node. So rainfall has to be transferred from the grid to the well locations, and the method matters.

**Why not snap each well to the nearest node.** One degree of latitude is about 110.6 km, so a 0.25 degree cell is roughly 27.7 km across. Snapping to the nearest node displaces a well by at most half a cell on each axis, which measures out at 10.2 km on average and 19.4 km at worst for our network. That sounds small, but it is a step function: a well 1 km inside a boundary takes 100 percent of one cell and nothing from the neighbour 1 km away. Measured against bilinear interpolation over the full record, nearest-node snapping shifts monthly rainfall at the wells by about 10 percent of the mean. For a project whose entire claim is that the unexplained part of the level is extraction, a 10 percent error in the only observed driver is not acceptable.

**What we use instead.** Rainfall is sampled at each well's true coordinates by bilinear interpolation over the four surrounding grid nodes. This removes the discretisation error and is consistent with how the source field was built, which matters for the next point.

**How precise can any method be here.** The IMD product is not a measurement grid. Pai et al. (2014) construct it by inverse distance weighting from a minimum of one and a maximum of four nearest stations within a 1.5 degree search radius, drawing on about 2600 gauges per day nationally. The 0.25 degree spacing is therefore a presentation resolution, not an information resolution, and the underlying field is smooth by construction. Two consequences follow. Sampling that smooth field smoothly, which is what bilinear does, is the methodologically consistent choice. And no method can recover true point rainfall at a wellhead from this product, so the honest claim is a well-centred estimate of a regionally smoothed field, not a point measurement.

**Why the well location matters at all.** 92.5 percent of these wells are 30 m deep or shallower, dug wells have a median depth of 11.1 m, and 2575 of 2759 sit in unconfined aquifers. These are shallow phreatic wells that respond to local recharge within a season, which is exactly the regime where spatial precision matters most. Section 11 tests how much the choice actually changes the result.

*Reference: Pai, D.S., Sridhar, L., Rajeevan, M., Sreejith, O.P., Satbhai, N.S. and Mukhopadyay, B. (2014). Development of a new high spatial resolution (0.25 x 0.25) long period (1901-2010) daily gridded rainfall data set over India. MAUSAM 65(1), 1-18.*

In [ ]:
GRID_NC   = PROC / 'india_monthly_grid.nc'
WELL_RAIN = PROC / 'well_monthly_rain_1998_2022.parquet'

wells_raw = pd.read_csv(CGWB)
wells_raw['well_id'] = np.arange(len(wells_raw))     # Station Code is unusable, see section 3

if not (WELL_RAIN.exists() and GRID_NC.exists()):
    mon, wet = [], []
    for f in sorted(IMDDIR.glob('imd_rf25_*.nc')):
        ds = xr.open_dataset(f)
        da = ds['RAINFALL']
        mon.append(da.resample(TIME='1MS').sum(skipna=True))
        wet.append((da > 2.5).resample(TIME='1MS').sum())    # IMD rainy day threshold
        ds.close()
    grid = xr.concat(mon, dim='TIME').rename('rain_mm')
    grid = grid.where(grid >= 0)                             # negatives are fill values
    wgrid = xr.concat(wet, dim='TIME').rename('wet_days')
    xr.Dataset({'rain_mm': grid, 'wet_days': wgrid}).to_netcdf(GRID_NC)

    wlat = xr.DataArray(wells_raw['Latitude'].values, dims='w')
    wlon = xr.DataArray(wells_raw['Longitude'].values, dims='w')
    rb = grid.interp(LATITUDE=wlat, LONGITUDE=wlon, method='linear').values
    wb = wgrid.interp(LATITUDE=wlat, LONGITUDE=wlon, method='linear').values
    months = pd.to_datetime(grid.TIME.values)
    (pd.DataFrame(rb, index=months, columns=wells_raw['well_id']).stack().rename('rain_mm')
       .to_frame()
       .join(pd.DataFrame(wb, index=months, columns=wells_raw['well_id']).stack().rename('wet_days'))
       .rename_axis(['month','well_id']).reset_index()
       .to_parquet(WELL_RAIN, index=False))

grid_ds = xr.open_dataset(GRID_NC)
well_rain = pd.read_parquet(WELL_RAIN)
well_rain['month'] = pd.to_datetime(well_rain['month'])
print(f'well-month rainfall: {well_rain.shape[0]:,} rows '
      f'({well_rain.well_id.nunique()} wells x {well_rain.month.nunique()} months)')
well_rain.head()

## 3. Data exploration

In [ ]:
print('shape:', wells_raw.shape)
season_cols = [c for c in wells_raw.columns if c[:3] in ('Jan','May','Aug','Nov') and '-' in c]
print(len(season_cols), 'seasonal reading columns (4 campaigns x 23 years)')
print('states:', wells_raw['State'].nunique(), '| districts:', wells_raw['District'].nunique())
print(wells_raw['Type of Well'].value_counts().to_dict())
print()
print('well depth by type (m):')
print(wells_raw.groupby('Type of Well')['Well Depth'].median().round(1).to_dict())
print('wells 30 m or shallower: %.1f%%' % ((wells_raw['Well Depth'] <= 30).mean()*100))
print('unconfined aquifer:', int((wells_raw['Aquifer Type'] == 'Unconfined').sum()), 'of', len(wells_raw))
print()
# Station Code is stored in scientific notation upstream and collides badly,
# so it cannot be a key; well_id (row index) is used instead.
print('unique Station Code:', wells_raw['Station Code'].nunique(), 'for', len(wells_raw), 'wells')

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 8))
sc = ax.scatter(wells_raw['Longitude'], wells_raw['Latitude'], s=6,
                c=wells_raw[season_cols].mean(axis=1), cmap='RdYlBu_r', vmin=0, vmax=20)
plt.colorbar(sc, label='mean water level depth 2000-22 (m below ground)')
ax.set_title('CGWB wells: mean groundwater depth')
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
plt.tight_layout(); plt.show()

Deep water tables cluster in the northwest and interior peninsular districts, shallow ones along the Gangetic plain and coasts. Coverage is dense in central and south India and sparse in the northeast, so results only generalize to monitored districts. The network is overwhelmingly shallow dug wells in unconfined aquifers, which respond to rainfall within a season.

## 4. Data preparation

Melt the wide table to one row per (well, campaign, year). Target variable: `gwl_mbgl`, depth to water in metres below ground level. Larger value = deeper water table = decline.

In [ ]:
idv = ['well_id','State','District','Latitude','Longitude','Type of Well','Well Depth','Reference_Sy']
long = wells_raw.melt(id_vars=idv, value_vars=season_cols, var_name='season_col', value_name='gwl_mbgl')
mon = {'Jan':1,'May':5,'Aug':8,'Nov':11}
long['season']    = long['season_col'].str[:3]
long['month_num'] = long['season'].map(mon)
long['year']      = 2000 + long['season_col'].str[-2:].astype(int)
long['date']      = pd.to_datetime(dict(year=long['year'], month=long['month_num'], day=1))
print(f'{long.shape[0]:,} rows, observed: {long.gwl_mbgl.notna().sum():,} ({long.gwl_mbgl.notna().mean()*100:.1f}%)')

# districts entering any district level analysis need at least 3 wells, applied once here
# so that every later district result uses the same population
wells_per_district = (long.dropna(subset=['gwl_mbgl'])
                          .groupby(['State','District'])['well_id'].nunique())
KEEP = wells_per_district[wells_per_district >= 3].index
print(f'districts with at least 3 wells: {len(KEEP)} of {wells_per_district.shape[0]}')
long[['well_id','State','District','season','year','gwl_mbgl']].head()

## 5. Missing data analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
long.groupby('year')['gwl_mbgl'].apply(lambda s: s.isna().mean()*100).plot.bar(ax=axes[0], color='#8172b2')
axes[0].set_title('% missing by year'); axes[0].set_ylabel('%')
long.groupby('season')['gwl_mbgl'].apply(lambda s: s.isna().mean()*100).reindex(['Jan','May','Aug','Nov']).plot.bar(ax=axes[1], color='#ccb974')
axes[1].set_title('% missing by campaign')
plt.tight_layout(); plt.show()

Decisions taken:

1. May is the weakest campaign (about 30% missing, many dug wells run dry before the monsoon). 2012, 2016 and 2020-21 are weak years. All four campaigns are kept but models will be evaluated per season.
2. No imputation of the target. Imputing levels would fabricate the signal we later interpret as extraction. Missing target rows are dropped.
3. Rainfall features have no gaps since the IMD grid is complete.
4. District level work requires at least 3 wells per district, which keeps 272 of 365 districts. This filter is applied once in section 4 so that the exploratory district results and the modelling table describe the same population.

## 6. Rainfall analysis

In [ ]:
well_rain['cal_month'] = well_rain['month'].dt.month
well_rain['year'] = well_rain['month'].dt.year
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
well_rain.groupby('cal_month')['rain_mm'].mean().plot.bar(ax=axes[0], color='#4c72b0')
axes[0].set_title('Mean monthly rainfall at well locations'); axes[0].set_xlabel('month'); axes[0].set_ylabel('mm')
yearly = well_rain.groupby('year')['rain_mm'].mean() * 12
yearly.loc[2000:].plot(ax=axes[1], marker='o', color='#55a868')
axes[1].axhline(yearly.loc[2000:].mean(), ls='--', c='gray')
axes[1].set_title('Mean annual rainfall, year to year'); axes[1].set_ylabel('mm/yr')
plt.tight_layout(); plt.show()
jjas_share = well_rain[well_rain.cal_month.isin([6,7,8,9])]['rain_mm'].sum() / well_rain['rain_mm'].sum()
print(f'JJAS monsoon share of annual rainfall: {jjas_share*100:.1f}%')

## 7. Groundwater level analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
long['gwl_mbgl'].hist(bins=60, ax=axes[0], color='#4c72b0')
axes[0].set_title(f'Target distribution (skew={long.gwl_mbgl.skew():.2f})'); axes[0].set_xlabel('m below ground')
np.log1p(long['gwl_mbgl']).hist(bins=60, ax=axes[1], color='#55a868')
axes[1].set_title('log1p(target)'); axes[1].set_xlabel('log1p(m)')
sns.boxplot(data=long, x='season', y='gwl_mbgl', order=['Jan','May','Aug','Nov'],
            showfliers=False, ax=axes[2], color='#ccb974')
axes[2].set_title('Seasonal cycle')
plt.tight_layout(); plt.show()

In [ ]:
# linear trend of post-monsoon (Nov) depth, districts with at least 3 wells only
nov = long[(long.season=='Nov') & long.gwl_mbgl.notna()].set_index(['State','District']).loc[KEEP].reset_index()
dmean = nov.groupby(['State','District','year'])['gwl_mbgl'].mean().reset_index()
def slope(g):
    return np.polyfit(g['year'], g['gwl_mbgl'], 1)[0] if len(g) >= 15 else np.nan
trend = dmean.groupby(['State','District']).apply(slope, include_groups=False).dropna().rename('slope_m_per_yr')
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
trend.hist(bins=50, ax=axes[0], color='#4c72b0'); axes[0].axvline(0, c='k')
axes[0].set_title(f'Nov depth trend, {len(trend)} districts ({(trend>0).mean()*100:.0f}% deepening)')
axes[0].set_xlabel('m/yr (positive = declining groundwater)')
trend.sort_values(ascending=False).head(12)[::-1].plot.barh(ax=axes[1], color='#c44e52')
axes[1].set_title('Fastest declining districts'); axes[1].set_xlabel('m/yr')
plt.tight_layout(); plt.show()

The target is right skewed, so models will be checked on both raw and log scales. Depth peaks in May and is shallowest in Aug, matching the monsoon cycle.

The trend chart uses only districts with at least 3 monitored wells. This matters: several districts that are famous extraction hotspots, including Kurukshetra, Ludhiana and Hisar, carry a single well each in this network and are therefore excluded rather than ranked on one well. The districts that remain at the top, such as Bathinda and Faridkot in the Punjab paddy belt and Bangalore Urban, are consistent with known over-extraction, which is a useful check before any model exists. A raw trend still cannot separate rain driven decline from pumping, and that separation is the point of the project.

## 8. Feature engineering

A water table sits where it does because of how much water went in and how much came out. Rainfall is the only inflow we observe, so the features below are all ways of describing recent rainfall at the well, plus a little static context. Four ideas drive the choices.

**The leakage rule.** Every feature must be computable on or before the date of the reading. All rainfall windows look backwards, the monsoon feature uses only the last completed monsoon, and the lag uses last year. This matters more here than in an ordinary prediction task, because we later read the model's leftover error as extraction pressure. If information from after the reading leaked in, the model would explain away part of that signal and the attribution would break with no visible symptom.

**The response time is unknown, so supply several.** An aquifer behaves like a bucket with a slow drain: accumulated rainfall matters, not today's rainfall. But a shallow dug well responds within a season while deeper storage integrates several years. Rather than guess, we supply totals over 3, 6, 12, 24 and 36 months and let the model decide.

**How the rain arrives matters, not only how much.** The same 100 mm soaks in over twenty gentle days and largely runs off in one afternoon. A count of rainy days above IMD's 2.5 mm threshold is a cheap proxy for that difference.

**A dry year against a dry place.** Raw rainfall mostly encodes geography, and that is already reflected in a district's water table. Attribution needs the time dimension, so the z score compares each well's rainfall against its own long run normal for the same calendar month. Minus 1.5 means unusually dry for that location, whichever location it is.

| Feature | What it captures |
|---|---|
| rain_3m, rain_6m | fast recharge response of shallow dug wells |
| rain_12m | one full hydrological year of recharge |
| rain_24m, rain_36m | carry over storage and multi year droughts |
| wetdays_12m | how the rain arrived, soaking against running off |
| rain_12m_z | dry year against dry place |
| last_monsoon_mm, last_monsoon_z | the last completed monsoon, the main recharge pulse |
| gwl_lag1y | aquifer memory, same well and season one year earlier |
| Reference_Sy, Well Depth, lat/lon | static hydrogeological setting |

In [ ]:
R = well_rain.pivot(index='month', columns='well_id', values='rain_mm').sort_index()
W = well_rain.pivot(index='month', columns='well_id', values='wet_days').sort_index()

feat = {}
for w in (3, 6, 12, 24, 36):
    feat[f'rain_{w}m'] = R.rolling(w, min_periods=w).sum()
feat['wetdays_12m'] = W.rolling(12, min_periods=12).sum()

r12 = feat['rain_12m']
cm = r12.groupby(r12.index.month)
feat['rain_12m_z'] = (r12 - cm.transform('mean')) / cm.transform('std').replace(0, np.nan)

jj = R[R.index.month.isin([6,7,8,9])]
jjas = jj.groupby(jj.index.year).sum(min_count=4)
jjas_z = (jjas - jjas.mean()) / jjas.std()

FEATS = list(feat.keys())
F = pd.concat([v.stack().rename(k) for k, v in feat.items()], axis=1).reset_index()
F.columns = ['date', 'well_id'] + FEATS

obs = long.merge(F, on=['well_id','date'], how='left')
# the Aug reading falls inside the ongoing monsoon, so its last completed JJAS is the previous year's
obs['monsoon_year'] = np.where(obs['month_num'].isin([1, 5, 8]), obs['year'] - 1, obs['year'])
jm = jjas.stack().rename('last_monsoon_mm').reset_index()
jm.columns = ['monsoon_year', 'well_id', 'last_monsoon_mm']
jz = jjas_z.stack().rename('last_monsoon_z').reset_index()
jz.columns = ['monsoon_year', 'well_id', 'last_monsoon_z']
obs = obs.merge(jm.merge(jz, on=['monsoon_year','well_id']),
                on=['well_id','monsoon_year'], how='left')

obs = obs.sort_values(['well_id','season','year'])
obs['gwl_lag1y'] = obs.groupby(['well_id','season'])['gwl_mbgl'].shift(1)
print(f'observation table: {obs.shape[0]:,} rows x {obs.shape[1]} cols')
obs[['well_id','District','season','year','gwl_mbgl','rain_12m','rain_12m_z','last_monsoon_z','gwl_lag1y']].head()

## 9. Correlation analysis

In [ ]:
FEAT_ALL = FEATS + ['last_monsoon_mm','last_monsoon_z','gwl_lag1y','Well Depth','Reference_Sy']
corr = obs[FEAT_ALL + ['gwl_mbgl']].corr(method='spearman')
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax, cbar_kws={'label': 'Spearman rho'})
ax.set_title('Feature vs target correlation (pooled)')
plt.tight_layout(); plt.show()

In [ ]:
# pooled vs within-district correlation
d = obs.dropna(subset=['gwl_mbgl'])
pooled = d['rain_12m_z'].corr(d['gwl_mbgl'])
within = (d.groupby(['State','District'])
            .apply(lambda g: g['rain_12m_z'].corr(g['gwl_mbgl']) if len(g) > 30 else np.nan, include_groups=False)
            .dropna())
fig, ax = plt.subplots(figsize=(9, 3.8))
within.hist(bins=40, ax=ax, color='#4c72b0'); ax.axvline(0, c='k')
ax.axvline(within.mean(), c='#c44e52', ls='--', label=f'within-district mean = {within.mean():.2f}')
ax.axvline(pooled, c='#55a868', ls=':', lw=2, label=f'pooled = {pooled:.2f}')
ax.set_title(f'corr(12-month rainfall anomaly, depth): {(within<0).mean()*100:.0f}% of districts negative')
ax.legend(); plt.tight_layout(); plt.show()

Observations.

**The sign is a sanity check on the merge as much as on the physics.** Rain raises the water table, and since the target is a depth measured downward, raised means a smaller number. Every rainfall feature should therefore correlate negatively, and every one does, most strongly over 3 to 6 months. That short response time is consistent with a network of shallow dug wells in unconfined aquifers. A positive sign here would have pointed at a broken join rather than at surprising hydrology.

**Spearman rather than Pearson**, because the target is right skewed and the relationships need not be linear. Spearman correlates ranks, so it measures whether the relationship is consistently in one direction and is not dominated by the tail of very deep wells.

**Pooled and within-district correlations answer different questions.** Two relationships link rain to level. Across space, wetter regions have shallower tables. Across time, a wetter year lifts a given district's table. Pooling every row blends them and gives a weaker number than computing the correlation inside each district. The pooled figure is contaminated because geology, well depth and extraction also vary across regions. Attribution is a within-district question, so the models must be district aware and cross validated by district.

**The lagged level is the strongest predictor and the most dangerous one.** Rank correlation is about 0.9 because aquifers have memory, but last year's level is the outcome of past rainfall and past pumping together. Putting it into the rainfall-expected model gives that model a shortcut: it can predict well without really using rainfall, the residual shrinks toward zero, and the extraction signal disappears into the prediction instead of remaining as error. So two variants are planned, rainfall only for attribution and plus lag for forecasting, and they are never mixed.

On correlation against causation: rainfall correlating with levels does not prove the unexplained decline is pumping. Irrigation demand itself rises in dry years, canals and cropping choices confound districts, and wells sample the aquifer non randomly. The mitigation is to model the rainfall only expectation, interpret only persistent multi year residuals as extraction pressure, and validate the resulting ranking against CGWB stress categories the models never see. The claim is attribution consistent with independent evidence, not proof of causation.

## 10. Feature relevance and sampling

Correlation only detects relationships that move consistently in one direction. Two further checks confirm the engineered features carry signal beyond that. Mutual information asks how much knowing a feature reduces uncertainty about the target without assuming any shape of relationship, and is zero only under genuine independence. Random forest importance gives a second, model based opinion by measuring each feature's contribution to tree splits. Where two methods built on different principles agree, the ranking can be trusted. The forest here is a 60 tree screening tool, not the project's model.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import mutual_info_regression

Xy = obs.dropna(subset=['gwl_mbgl'] + FEAT_ALL[:-2])[FEAT_ALL + ['gwl_mbgl','season']].dropna()
X = pd.get_dummies(Xy.drop(columns='gwl_mbgl'), columns=['season'])
y = Xy['gwl_mbgl']
mi = pd.Series(mutual_info_regression(X, y, random_state=RNG, n_neighbors=5), index=X.columns)
rf = RandomForestRegressor(n_estimators=60, max_depth=14, n_jobs=-1, random_state=RNG).fit(X, y)
fi = pd.Series(rf.feature_importances_, index=X.columns)
pd.DataFrame({'mutual_info': mi, 'rf_importance': fi}).sort_values('rf_importance') \
  .plot.barh(figsize=(9, 6), color=['#4c72b0', '#c44e52'])
plt.title('Mutual information vs random forest importance')
plt.tight_layout(); plt.show()

Both measures agree. `gwl_lag1y` dominates, which is expected given aquifer memory. Among rainfall only features the 6 to 12 month windows and the anomaly z scores rank highest. Nothing is dropped at this stage: the screen exists to justify keeping all the windows rather than to prune, and the regularised linear baseline will handle the overlap between correlated windows itself.

On over and undersampling: it does not apply to the regression target, because resampling is a classification technique. The target's skew is handled by evaluating on the log scale alongside the raw scale. Resampling becomes relevant in the classification validation step, where CGWB stress categories are roughly 67% safe, 15% semi-critical, 4% critical and 14% over-exploited nationally. The plan there is class weights first, SMOTE and random undersampling as comparisons, stratified grouped splits, and PR-AUC instead of accuracy.

## 11. Spatial aggregation sensitivity

Section 2 argued for bilinear interpolation on methodological grounds. This section tests empirically how much the choice actually changes the answer, by running the whole chain under four ways of transferring rainfall from the grid to a well:

1. **nearest node**, snapping each well to the closest 0.25 degree lattice point
2. **bilinear**, the method adopted, sampling at the well's true coordinates
3. **25 km buffer**, averaging every grid cell whose centre lies within 25 km of the well
4. **50 km buffer**, the same at 50 km

The buffer variants are not arbitrary. A shallow phreatic well responds to recharge over an area rather than to rain falling on the wellhead, so an area average is a physically reasonable alternative rather than a degraded one. Comparing across all four answers the real question, which is not whether the rainfall numbers differ but whether the district ranking that the project produces depends on the choice.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

GRID = grid_ds['rain_mm']
MONTHS = pd.to_datetime(GRID.TIME.values)
wlat_v, wlon_v = wells_raw['Latitude'].values, wells_raw['Longitude'].values

def snap(v, o, s=0.25):
    return np.round((v - o) / s) * s + o

methods = {}
methods['bilinear'] = R.values                                   # already built in section 8
methods['nearest_node'] = GRID.sel(
    LATITUDE=xr.DataArray(snap(wlat_v, 6.5), dims='w'),
    LONGITUDE=xr.DataArray(snap(wlon_v, 66.5), dims='w'), method='nearest').values

gla, glo = np.meshgrid(GRID.LATITUDE.values, GRID.LONGITUDE.values, indexing='ij')
gla_f, glo_f = gla.ravel(), glo.ravel()
Rg = GRID.values.reshape(len(MONTHS), -1)
ok_cell = np.isfinite(Rg).mean(0) > 0.5
for radius in (25.0, 50.0):
    out = np.full((len(MONTHS), len(wells_raw)), np.nan, dtype='float32')
    for i in range(len(wells_raw)):
        dy = (gla_f - wlat_v[i]) * 110.6
        dx = (glo_f - wlon_v[i]) * 110.6 * np.cos(np.radians(wlat_v[i]))
        m = (dy**2 + dx**2 <= radius**2) & ok_cell
        if m.any():
            out[:, i] = np.nanmean(Rg[:, m], axis=1)
    methods[f'buffer_{int(radius)}km'] = out

base = methods['bilinear']
ok0 = np.isfinite(base)
print('monthly rainfall at wells, each method against bilinear:')
for k, v in methods.items():
    o = ok0 & np.isfinite(v)
    print(f'  {k:<14} r={np.corrcoef(v[o], base[o])[0,1]:.4f}  '
          f'MAE={np.abs(v[o]-base[o]).mean():6.2f} mm  '
          f'({np.abs(v[o]-base[o]).mean()/base[o].mean()*100:5.1f}% of mean)')

In [ ]:
def rank_districts(mat):
    """rainfall-only ridge with district effects; rank districts by the trend in the residual."""
    df = pd.DataFrame(mat, index=MONTHS, columns=wells_raw['well_id'].values)
    f = {f'rain_{w}m': df.rolling(w, min_periods=w).sum() for w in (3, 6, 12, 24, 36)}
    r12v = f['rain_12m']; cmv = r12v.groupby(r12v.index.month)
    f['rain_12m_z'] = (r12v - cmv.transform('mean')) / cmv.transform('std').replace(0, np.nan)
    f['wetdays_12m'] = W.rolling(12, min_periods=12).sum()
    cols = list(f.keys())
    Fm = pd.concat([v.stack().rename(k) for k, v in f.items()], axis=1).reset_index()
    Fm.columns = ['date','well_id'] + cols

    o = (long.dropna(subset=['gwl_mbgl']).merge(Fm, on=['well_id','date'], how='left')
              .set_index(['State','District']).loc[KEEP].reset_index())
    t = o.groupby(['State','District','year','season']).agg(
            gwl_mean=('gwl_mbgl','mean'), **{c: (c,'mean') for c in cols}).reset_index()
    t = t.dropna(subset=cols + ['gwl_mean'])

    Xd = pd.get_dummies(t[cols + ['season']], columns=['season'], dtype=float)
    Dd = pd.get_dummies(t['State'] + '|' + t['District'], dtype=float)
    XX = np.hstack([StandardScaler().fit_transform(Xd), Dd.values])
    t['resid'] = t['gwl_mean'].values - Ridge(alpha=1.0).fit(XX, t['gwl_mean'].values).predict(XX)
    return (t.groupby(['State','District'])
              .apply(lambda g: np.polyfit(g['year'], g['resid'], 1)[0]
                     if g['year'].nunique() >= 15 else np.nan, include_groups=False).dropna())

ranks = pd.DataFrame({k: rank_districts(v) for k, v in methods.items()}).dropna()
print(f'districts ranked under every method: {len(ranks)}\n')
print('Spearman agreement between district rankings:')
print(ranks.corr(method='spearman').round(4).to_string())

top = ranks['bilinear'].rank(ascending=False).nsmallest(20).index
for k in ('nearest_node','buffer_25km','buffer_50km'):
    ov = len(set(top) & set(ranks[k].rank(ascending=False).nsmallest(20).index))
    print(f'top-20 overlap, bilinear vs {k}: {ov}/20')

Result. The rainfall values themselves differ materially between methods, with nearest node about 10 percent away from bilinear on monthly totals and the 50 km buffer further still. The district ranking, which is what the project actually produces, does not: every pair of methods agrees at a Spearman correlation above 0.98, and the top twenty districts are almost identical across methods.

Two conclusions follow, and both matter. Bilinear is adopted because it is the defensible choice on methodological grounds, removing a 10 percent discretisation error for no cost. And the headline finding is robust: the districts flagged as having a widening gap between observed and rainfall-expected levels are not an artefact of how rainfall was transferred from the grid. The remaining spatial uncertainty is dominated by the IMD product's own construction, described in section 2, not by our extraction step.

## 12. Final data set

The earlier sections combine into the modelling table. Cleanup decided which rows are trustworthy, feature engineering decided what each row knows about preceding rainfall, the relevance screen confirmed those features carry signal, and the sensitivity test confirmed the spatial merge does not drive the result. The remaining decision is the grain, meaning what one row represents.

Two files are saved. `well_season_obs.parquet` keeps every individual well reading with its features, for later well level work. `final_dataset_district_season.csv` is the modelling table, one row per district per campaign.

In [ ]:
obs.to_parquet(PROC / 'well_season_obs.parquet', index=False)

valid = obs[obs.gwl_mbgl.notna()].set_index(['State','District']).loc[KEEP].reset_index()
final = (valid.groupby(['State','District','year','season','date'])
   .agg(gwl_mean=('gwl_mbgl','mean'), gwl_median=('gwl_mbgl','median'), gwl_std=('gwl_mbgl','std'),
        n_wells=('well_id','nunique'), lat=('Latitude','mean'), lon=('Longitude','mean'),
        well_depth_mean=('Well Depth','mean'), sy_mean=('Reference_Sy','mean'),
        rain_3m=('rain_3m','mean'), rain_6m=('rain_6m','mean'), rain_12m=('rain_12m','mean'),
        rain_24m=('rain_24m','mean'), rain_36m=('rain_36m','mean'),
        wetdays_12m=('wetdays_12m','mean'), rain_12m_z=('rain_12m_z','mean'),
        last_monsoon_mm=('last_monsoon_mm','mean'), last_monsoon_z=('last_monsoon_z','mean'),
        gwl_lag1y=('gwl_lag1y','mean'))
   .reset_index())
final.to_csv(PROC / 'final_dataset_district_season.csv', index=False)
print(f'final data set: {final.shape[0]:,} rows x {final.shape[1]} cols | '
      f'{final.groupby(["State","District"]).ngroups} districts | {final.year.min()}-{final.year.max()}')
final.head()

**What the table is.** One row per district per measurement campaign, for the districts with at least three wells, covering 2000 to 2022. The target is the mean depth across that district's wells. The median, the standard deviation and the well count travel alongside as quality indicators for each row.

**Why this grain.**

1. It matches the question. Each row is a district at a point in time with its preceding rainfall attached, which is the exact shape of the counterfactual the project asks: what level should this district have had, given this rainfall.
2. It matches the decision unit. Recharge interventions are planned and funded per district, so a district ranking is directly usable while a ranking of individual wells is not.
3. It keeps the history attribution needs. Twenty three years of four campaigns preserves both the seasonal cycle and the long term trend. Collapsing to annual figures would discard the seasonal contrast.
4. The target is stable. Requiring at least three wells stops one unusual well from defining a district, and n_wells and gwl_std make it possible to weight or filter rows later.
5. The size is practical. Roughly twenty thousand rows train in seconds, which matters when the plan calls for repeated cross validation across districts and years.

**Alternatives considered and rejected.** Using each district's trend slope as the target would leave only a few hundred rows and, more seriously, no counterfactual, since a slope cannot be asked what it would have been under different rainfall. Using the CGWB stress category as the target would turn the project into classification and collapse 23 years into a single static label, so that belongs in validation instead. Using the year on year change in level would remove the persistent component, which is precisely the part that carries the extraction signal.

**Known limitation.** The district mean is taken over whichever wells reported that campaign, and that set changes over time because of missing readings. A district whose shallow wells go dry in May and stop reporting will show a mean biased deep for compositional rather than hydrological reasons. The planned check is a well level model with well effects, and a balanced panel of wells with near complete records, compared against the district level result.

## 13. Rainfall time series model

Before modelling groundwater, it is worth checking how predictable rainfall itself is from its own history. This section builds a small statistical time series model on district level monthly rainfall: a SARIMA model for next-month prediction, compared against a monthly climatology baseline, and a check of how predictable next year's monsoon total is. Train period 1998-2020, test period 2021-2022.

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

dist_rain = (wells_raw[['well_id','State','District']]
             .merge(well_rain[['well_id','month','rain_mm']], on='well_id')
             .groupby(['State','District','month'])['rain_mm'].mean().reset_index())
print('district-month series:', dist_rain.shape)

def get_series(state, district):
    s = dist_rain[(dist_rain.State==state) & (dist_rain.District==district)]
    return s.set_index('month')['rain_mm'].asfreq('MS')

def mae(a, b):
    return (a - b).abs().mean()

y = get_series('Andhra pradesh', 'Anantapur')
train, test = y[:'2020-12'], y['2021-01':]
clim = train.groupby(train.index.month).mean()
clim_pred = pd.Series([clim[m] for m in test.index.month], index=test.index)

sar = SARIMAX(train, order=(1,0,1), seasonal_order=(1,1,1,12),
              enforce_stationarity=False, enforce_invertibility=False).fit(disp=0)
sar_pred = sar.forecast(len(test))

fig, ax = plt.subplots(figsize=(12, 4))
y['2017':].plot(ax=ax, label='actual', color='#4c72b0')
sar_pred.plot(ax=ax, label=f'SARIMA (MAE {mae(test, sar_pred):.0f} mm)', color='#c44e52')
clim_pred.plot(ax=ax, label=f'climatology (MAE {mae(test, clim_pred):.0f} mm)', color='#55a868', ls='--')
ax.axvline(pd.Timestamp('2021-01-01'), c='gray', ls=':')
ax.set_title('Anantapur monthly rainfall: next-month forecasts on the 2021-22 test window')
ax.set_ylabel('mm'); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
rng = np.random.RandomState(RNG)
keys = dist_rain.groupby(['State','District']).size().index.tolist()
sample = [keys[i] for i in rng.choice(len(keys), 24, replace=False)]

rows = []
for s, dd in sample:
    ys = get_series(s, dd)
    tr, te = ys[:'2020-12'], ys['2021-01':]
    cl = tr.groupby(tr.index.month).mean()
    clp = pd.Series([cl[m] for m in te.index.month], index=te.index)
    m_ = SARIMAX(tr, order=(1,0,1), seasonal_order=(1,1,1,12),
                 enforce_stationarity=False, enforce_invertibility=False).fit(disp=0)
    rows.append((s, dd, mae(te, m_.forecast(len(te))), mae(te, clp)))

res = pd.DataFrame(rows, columns=['State','District','sarima_mae','clim_mae'])
print(res[['sarima_mae','clim_mae']].mean().round(1))
print('SARIMA beats climatology in', (res.sarima_mae < res.clim_mae).sum(), 'of', len(res), 'districts')

In [ ]:
dist_rain['y'] = dist_rain.month.dt.year
dist_rain['m'] = dist_rain.month.dt.month
jjas_d = (dist_rain[dist_rain.m.isin([6,7,8,9])]
          .groupby(['State','District','y'])['rain_mm'].sum().reset_index())

ac1 = (jjas_d.groupby(['State','District'])['rain_mm']
       .apply(lambda s: s.autocorr(1) if len(s) == 25 else np.nan).dropna())

errs_clim, errs_pers = [], []
for (s, dd), g in jjas_d.groupby(['State','District']):
    g = g.sort_values('y').set_index('y')['rain_mm']
    if len(g) < 25: continue
    for yy in range(2018, 2023):
        errs_clim.append(abs(g[yy] - g[g.index < yy].mean()))
        errs_pers.append(abs(g[yy] - g[yy-1]))

fig, ax = plt.subplots(figsize=(9, 3.6))
ac1.hist(bins=40, ax=ax, color='#4c72b0'); ax.axvline(0, c='k')
ax.axvline(ac1.mean(), c='#c44e52', ls='--', label=f'mean = {ac1.mean():.2f}')
ax.set_title('Lag-1 autocorrelation of annual JJAS totals per district')
ax.legend(); plt.tight_layout(); plt.show()
print(f'year-ahead JJAS MAE, 2018-2022: climatology {np.mean(errs_clim):.0f} mm | persistence (last year) {np.mean(errs_pers):.0f} mm')

Takeaways from the time series experiment:

1. Next month: SARIMA only matches the climatology baseline. Monthly rainfall in India is almost entirely seasonality; once the seasonal cycle is accounted for there is very little extra month-to-month structure to exploit.
2. Next year: annual monsoon totals are close to unpredictable from rainfall history alone. Lag-1 autocorrelation averages about 0.1 and climatology beats last-year persistence. Skilful monsoon forecasts need external drivers such as ENSO or IOD indices, which are outside this project's data.
3. Consequence for the project design: since rainfall cannot be usefully forecast from itself, the expected rainfall in what-if scenarios will be the climatology, and scenarios will be expressed as percentage deviations from it. The groundwater model then answers what the level would be under that rainfall, which is exactly the counterfactual the residual design needs.

## 14. LLM scope

The LLM is planned as the interface on top of the trained models, not as a model itself:

1. Natural language what-if queries, parsed by a LangGraph agent into calls to the FastAPI endpoints (/predict, /whatif, /district_rank), with answers written only from returned numbers.
2. Auto generated one page district briefs combining trend, residual ranking and peer comparison.
3. RAG over CGWB assessment reports (chunking, embeddings, retrieval) to give the agent citable district context. This is also where the text mining topic enters the project.

## 15. Conclusion

1. Both datasets are merged at the well level, with rainfall transferred from the IMD grid by bilinear interpolation at each well's true coordinates rather than by snapping to the nearest node, which removes an error of about 10 percent of mean monthly rainfall.
2. The merge choice was tested rather than assumed. Four spatial aggregation schemes, from nearest node to a 50 km recharge zone average, produce district rankings that agree at Spearman correlations above 0.98, so the headline result does not depend on it.
3. Data quality issues found and handled: Station Code is not a usable key and was replaced with well_id, 13.6% of target readings are missing and concentrated in May campaigns and in 2012, 2016 and 2020-21, the target is never imputed, and district level work is restricted to the districts with at least three wells so that every district result describes the same population.
4. The expected physics is visible in the data: the monsoon delivers about 81% of annual rain, depth cycles May-deep and Aug-shallow, and rainfall anomalies correlate negatively with depth in the large majority of districts.
5. Features are defined with an explicit no-leakage rule and screened with mutual information and a random forest, with the 6 to 12 month rainfall windows and the anomaly z scores leading among rainfall features.
6. A small SARIMA model shows monthly rainfall is essentially seasonality and next year's monsoon total is close to unpredictable from rainfall history alone, so what-if scenarios will be framed as percentage deviations from climatology.